# Charger booking protocol with `Hold`

This notebook updates the earlier contract-repair prototype around requests of the form

$$
\langle \top \mid \exists_I H^d(j=k) \rangle.
$$

Protocol:

1. **Probe-strengthen** the flexible request:
   $$
   \exists_I H^d p \mapsto \forall_I H^d p
   \equiv \forall_{I \oplus [0,d]} p.
   $$
2. Merge the strengthened request with current commitments.
3. If the strengthened merge succeeds, generate concrete `ForAll` booking offers directly from the original request.
4. Otherwise use `weaken -> separate -> merge -> postprocess/project` to expose the surviving available region.
5. **Concretize** that region into concrete `ForAll` offers.

The two strengthenings have different roles:

- `probe_strengthen`: tests the preferred temporal envelope.
- `concretize`: selects commitment-ready concrete intervals.

**API assumption:** `interval_logic` exposes `Hold(duration, expression)`, and simplification knows
`ForAll(I, Hold(d, p)) == ForAll(I ⊕ [0,d], p)`.


In [1]:

from dataclasses import dataclass
from contextlib import contextmanager

import portion as P
import interval_logic as il


@dataclass
class Contract:
    assumption: il.Expression | str
    guarantee: il.Expression | str

    def __post_init__(self) -> None:
        self.assumption = il.as_expr(self.assumption).simplify()

        # Maintain saturated-form invariant.
        self.guarantee = il.Or((
            il.as_expr(self.guarantee),
            il.Not(self.assumption),
        )).simplify()

    @staticmethod
    def _s(
        assumption: il.Expression | str,
        guarantee: il.Expression | str,
    ) -> str:
        return f"⟨ {il.as_expr(assumption)} ∣ {il.as_expr(guarantee)} ⟩"

    def __str__(self) -> str:
        return self._s(self.assumption, self.guarantee)

    @staticmethod
    def merge_s(self: "Contract", other: "Contract") -> str:
        return f"{self!s} ⊗ {other!s}"

    def merge(self, other: "Contract") -> "Contract":
        return Contract(
            il.And((self.assumption, other.assumption)),
            il.Or((
                il.And((self.guarantee, other.guarantee)),
                il.Not(il.And((self.assumption, other.assumption))),
            )),
        )

    @staticmethod
    def separate_s(self: "Contract", other: "Contract") -> str:
        return f"{self!s} ⦼ {other!s}"

    def separate(self, other: "Contract") -> "Contract":
        return Contract(
            il.Or((
                il.And((self.assumption, other.guarantee)),
                il.Not(il.And((self.guarantee, other.assumption))),
            )),
            il.And((self.guarantee, other.assumption)),
        )

    def to_nnf(self) -> "Contract":
        return Contract(
            il.to_nnf(self.assumption),
            il.to_nnf(self.guarantee),
        )

    def simplify(self) -> "Contract":
        return Contract(
            self.assumption.simplify(),
            self.guarantee.simplify(),
        )

    def replace(self, *fns) -> "Contract":
        return Contract(
            self.assumption.replace(*fns),
            self.guarantee.replace(*fns),
        )


@contextmanager
def step(msg: str, disable: int = 4):
    counter = list(getattr(step, "counter", (0,)))
    counter[-1] += 1

    if len(counter) == 1:
        println(f"⏺ Step {counter[0]}: {msg}")
    else:
        println(
            f"⏺ Step {'.'.join(map(str, counter))}: {msg}",
            marker=["├", "│"],
        )

    step.counter = tuple(counter + [0])
    _disable = getattr(step, "disable", 4)
    step.disable = disable

    try:
        yield
    finally:
        pass

    step.disable = _disable
    step.counter = step.counter[:-1]

    if len(step.counter) == 1:
        println("╯", left=0)
    else:
        println("", marker=["├─╯", "│"])


def reset_step():
    step.counter = (0,)
    step.disable = 4


def println(*lines: str, **kwds):
    assert all(isinstance(line, str) for line in lines)

    lines = [""] if len(lines) == 0 else list(lines)
    marker = kwds.get("marker", "│")

    top = kwds.get("top", kwds.get("y", 0))
    bottom = kwds.get("bottom", kwds.get("y", 0))
    left = kwds.get("left", kwds.get("x", 0))
    right = kwds.get("right", kwds.get("x", 0))

    indent = len(getattr(step, "counter", (0,))) - 1
    if indent >= getattr(step, "disable", 4):
        return

    ms = [marker] if isinstance(marker, str) else list(marker)
    ms = [m + " " for m in ms]
    ms += [ms[-1]] * (indent - len(ms))
    ws = "".join(reversed(ms[:indent]))

    max_width = max(len(line) for line in lines) + left + right
    lines = [
        ws + (" " * left + line + " " * right).ljust(max_width)
        for line in lines
    ]
    lines = [ws] * top + lines + [ws] * bottom

    print("\n".join(lines))


def printeq(arg, *args):
    lines = [str(arg)]
    lines += [f"= {value!s}" for value in args]
    println(*lines)


## Request and offer objects

`Request.interval` is the allowed **start-time interval**.

`Offer.interval` is a concrete reserved charging interval.


In [2]:

@dataclass(frozen=True)
class Request:
    job: int
    duration: float
    interval: P.Interval

    @property
    def proposition(self) -> il.Expression:
        return il.Signal("j").equals(self.job)

    @property
    def contract(self) -> Contract:
        return Contract(
            True,
            il.Exists(
                self.interval,
                il.Hold(self.duration, self.proposition),
            ),
        )


@dataclass(frozen=True)
class Offer:
    job: int
    duration: float
    interval: P.Interval

    @property
    def contract(self) -> Contract:
        return Contract(
            True,
            il.ForAll(
                self.interval,
                il.Signal("j").equals(self.job),
            ),
        )


## Protocol transformations

Relative to target job `k`, the projection forgets which *other* job occupies the charger:

- `j = k` becomes `p`,
- `j = n`, `n != k`, becomes `¬p`.

This is a scheduler projection, not a generic logical equivalence.


In [3]:

p = il.Atom("p")
j = il.Signal("j")


def project_job(job: int):
    target = il.Natural(job)

    def replacer(psi):
        match psi:
            case il.Equal(left, right):
                if left == j:
                    return p if right == target else il.Not(p)
                if right == j:
                    return p if left == target else il.Not(p)

        return psi

    return replacer


def probe_strengthen(psi):
    # ∃_I H^d p -> ∀_I H^d p
    # Simplification should lower the latter to ∀_(I ⊕ [0,d]) p.
    match psi:
        case il.Exists(I, il.Hold(d, phi)):
            return il.ForAll(I, il.Hold(d, phi)).simplify()

    return psi


def weaken(psi):
    # Repair relaxation inherited from the old prototype:
    # ∀_I p -> ∃_I p
    match psi:
        case il.ForAll(I, phi):
            return il.Exists(I, phi)

    return psi


def overlap(psi):
    # ∀_I ¬p ∧ ∃_J p -> ∀_I ¬p ∧ ∃_(J-I) p
    match psi:
        case il.And((il.ForAll(I, phi), il.Exists(J, q))):
            if phi == il.Not(q):
                return il.And((
                    il.ForAll(I, phi),
                    il.Exists(J - I, q),
                ))

        case il.And((il.Exists(J, q), il.ForAll(I, phi))):
            if phi == il.Not(q):
                return il.And((
                    il.ForAll(I, phi),
                    il.Exists(J - I, q),
                ))

    return psi


def postprocess_repair(C_ms: Contract, job: int) -> Contract:
    C_post = Contract(
        True,
        C_ms.guarantee & C_ms.assumption,
    )

    C_post = C_post.replace(project_job(job)).simplify()
    C_post = C_post.replace(overlap).simplify()

    return C_post


## Offer generation

This notebook uses a simple fixed-duration grid policy after the contract operations expose a usable interval. The policy stays outside the contract algebra.


In [4]:

def grid_windows(
    interval: P.Interval,
    duration: float,
    *,
    step_size: float = 1.0,
) -> list[P.Interval]:
    windows = []

    for component in interval:
        start = float(component.lower)
        stop = float(component.upper)

        while start + duration <= stop:
            windows.append(P.closedopen(start, start + duration))
            start += step_size

    return windows


def generate_offers(
    job: int,
    interval: P.Interval,
    duration: float,
    *,
    step_size: float = 1.0,
) -> list[Offer]:
    return [
        Offer(job, duration, J)
        for J in grid_windows(
            interval,
            duration,
            step_size=step_size,
        )
    ]


def find_available_interval(
    expression: il.Expression,
) -> P.Interval | None:
    found = None

    def visit(psi):
        nonlocal found

        if found is None:
            match psi:
                case il.Exists(I, phi) if phi == p:
                    found = I

        return psi

    expression.replace(visit)
    return found


# Case A — strengthened probe fails

Existing commitment:

\[
C^{cmt}
=
\langle \top \mid \forall_{[1,3)}(j=1) \rangle.
\]

Flexible request:

\[
C^{req}_2
=
\langle \top \mid \exists_{[2,4)} H^1(j=2) \rangle.
\]

For this toy notebook we explicitly follow the **failed-probe** branch so the repair algebra remains visible.


In [5]:

reset_step()

with step("Protocol — failed strengthened probe"):

    println("Commitments")
    printeq(
        C_cmt := Contract(True, "∀[1,3): j=1"),
        C_cmt_p := C_cmt.replace(project_job(2)),
    )

    println("Request", top=1)
    req = Request(
        job=2,
        duration=1.0,
        interval=P.closedopen(2, 4),
    )
    printeq(
        C_req := req.contract,
        C_req_p := C_req.replace(project_job(req.job)),
    )

    with step("Probe strengthen"):
        printeq(
            C_req_p,
            C_str := C_req_p.replace(probe_strengthen),
        )

    with step("Merge strengthened"):
        printeq(
            Contract.merge_s(C_cmt_p, C_str),
            C_probe := C_cmt_p.merge(C_str),
        )

        println(
            "Probe branch for this example: EMPTY",
            "→ continue to repair availability",
            top=1,
        )

    with step("Repair availability"):

        println("Weaken")
        printeq(
            C_str,
            C_w := C_str.replace(weaken),
        )

        println("Separate", top=1)
        printeq(
            Contract.separate_s(C_w, C_cmt_p),
            C_s := C_w.separate(C_cmt_p),
        )

        println("Merge separated", top=1)
        printeq(
            Contract.merge_s(C_s, C_cmt_p),
            C_ms := C_s.merge(C_cmt_p),
        )

        println("Postprocess / project", top=1)
        printeq(
            Contract._s(True, C_ms.guarantee & C_ms.assumption),
            C_post := postprocess_repair(C_ms, req.job),
        )

    with step("Concretize"):
        available = find_available_interval(C_post.guarantee)

        println(f"Available region: {available}")

        offers = generate_offers(
            req.job,
            available,
            req.duration,
            step_size=1.0,
        )

        for n, offer in enumerate(offers, 1):
            println(
                f"Offer {n}",
                str(offer.contract),
            )


⏺ Step 1: Protocol — failed strengthened probe
│ Commitments
│ ⟨ ⊤ ∣ ∀[1.0,3.0): j = 1 ⟩
│ = ⟨ ⊤ ∣ ∀[1.0,3.0): ¬p ⟩ 
│ 
│ Request
│ ⟨ ⊤ ∣ ∃[2,4): H^1 j = 2 ⟩
│ = ⟨ ⊤ ∣ ∃[2,4): H^1 p ⟩  
├ ⏺ Step 1.1: Probe strengthen
│ │ ⟨ ⊤ ∣ ∃[2,4): H^1 p ⟩  
│ │ = ⟨ ⊤ ∣ ∀[2,4): H^1 p ⟩
├─╯ 
├ ⏺ Step 1.2: Merge strengthened
│ │ ⟨ ⊤ ∣ ∀[1.0,3.0): ¬p ⟩ ⊗ ⟨ ⊤ ∣ ∀[2,4): H^1 p ⟩
│ │ = ⟨ ⊤ ∣ (∀[1.0,3.0): ¬p) ∧ (∀[2,4): H^1 p) ⟩  
│ │ 
│ │ Probe branch for this example: EMPTY
│ │ → continue to repair availability   
├─╯ 
├ ⏺ Step 1.3: Repair availability
│ │ Weaken
│ │ ⟨ ⊤ ∣ ∀[2,4): H^1 p ⟩  
│ │ = ⟨ ⊤ ∣ ∃[2,4): H^1 p ⟩
│ │ 
│ │ Separate
│ │ ⟨ ⊤ ∣ ∃[2,4): H^1 p ⟩ ⦼ ⟨ ⊤ ∣ ∀[1.0,3.0): ¬p ⟩           
│ │ = ⟨ (∀[1.0,3.0): ¬p) ∨ (∀[2,4): ¬H^1 p) ∣ ∃[2,4): H^1 p ⟩
│ │ 
│ │ Merge separated
│ │ ⟨ (∀[1.0,3.0): ¬p) ∨ (∀[2,4): ¬H^1 p) ∣ ∃[2,4): H^1 p ⟩ ⊗ ⟨ ⊤ ∣ ∀[1.0,3.0): ¬p ⟩
│ │ = ⟨ (∀[1.0,3.0): ¬p) ∨ (∀[2,4): ¬H^1 p) ∣ ∃[2,4): H^1 p ⟩                       
│ │ 
│ │ Postprocess / project
│ │ ⟨ ⊤ ∣ (∃[2,4): H^1 p) 

TypeError: 'NoneType' object is not iterable

# Case B — strengthened probe succeeds

Here the request starts after the existing commitment:

\[
C^{req}_2
=
\langle \top \mid \exists_{[3,5)} H^1(j=2) \rangle.
\]

If the strengthened probe is nonempty, the repair path is skipped and concrete offers are generated directly from the original flexible request.


In [ ]:

reset_step()

with step("Protocol — successful strengthened probe"):

    println("Commitments")
    printeq(
        C_cmt := Contract(True, "∀[1,3): j=1"),
        C_cmt_p := C_cmt.replace(project_job(2)),
    )

    println("Request", top=1)
    req = Request(
        job=2,
        duration=1.0,
        interval=P.closedopen(3, 5),
    )
    printeq(
        C_req := req.contract,
        C_req_p := C_req.replace(project_job(req.job)),
    )

    with step("Probe strengthen"):
        printeq(
            C_req_p,
            C_str := C_req_p.replace(probe_strengthen),
        )

    with step("Merge strengthened"):
        printeq(
            Contract.merge_s(C_cmt_p, C_str),
            C_probe := C_cmt_p.merge(C_str),
        )

        println(
            "Probe branch for this example: NONEMPTY",
            "→ skip repair",
            top=1,
        )

    with step("Concretize original request"):
        offers = generate_offers(
            req.job,
            req.interval,
            req.duration,
            step_size=1.0,
        )

        for n, offer in enumerate(offers, 1):
            println(
                f"Offer {n}",
                str(offer.contract),
            )


## Next iteration

Two pieces intentionally remain outside the contract algebra:

- **Nonemptiness / consistency:** replace the explicit branch labels with a scheduler satisfiability check over natural-number signals.
- **Offer policy:** `generate_offers` currently preserves duration on a fixed grid. Later policies can prioritize earliest start, maximum duration, partial duration/energy, etc.

The contract operations expose the temporal resource that survives; policy chooses the concrete commitment-ready `ForAll` contracts.
